In [1]:
import kagglehub
from pathlib import Path
import shutil

# create root dir for data
root = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out = root / "data" / "conll2003"
out.mkdir(parents=True, exist_ok=True)

# pull data from kaggle
path = kagglehub.dataset_download("juliangarratt/conll2003-dataset")
src = Path(path)
for n in ["eng.train", "eng.testa", "eng.testb"]:
    f = src / n if (src / n).exists() else next(src.rglob(n), None)
    if f:
        shutil.copy(f, out / n)

print("Original CoNLL-2003:", out)

Original CoNLL-2003: /home/arda/Documents/uva_msds/Sem4/DeepLearning/ds6050-project/data/conll2003


In [2]:
def parse_conll(filepath):
    """Read a CoNLL file and return a list of sentences.
    Each sentence is a list of (word, ner_tag) tuples."""
    sentences = []
    current = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            # skip DOCSTART and blank lines
            if line.startswith("-DOCSTART-"):
                continue
            if line == "":
                if current:
                    # end of sentence, add to list
                    sentences.append(current)
                    current = []
            else:
                # columns are: word, POS, chunk, NER
                parts = line.split()
                word = parts[0]
                ner  = parts[-1]
                current.append((word, ner))
        if current:
            sentences.append(current)
    return sentences

train_sentences = parse_conll(out / "eng.train")
dev_sentences   = parse_conll(out / "eng.testa")
test_sentences  = parse_conll(out / "eng.testb")

print(f"Train: {len(train_sentences)} sentences")
print(f"Dev:   {len(dev_sentences)} sentences")
print(f"Test:  {len(test_sentences)} sentences")

Train: 14041 sentences
Dev:   3250 sentences
Test:  3453 sentences


Note:
- Train: 14,987 sentences - 946 (DOCSTART_Count) = 14,041 without -DOCSTART-
- We need a `parse_conll_with_docs` function for the doc context models

In [3]:
set(tag for sentence in train_sentences for word, tag in sentence)

{'B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O'}

In [4]:
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", 
              "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
              
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

In [5]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset, DatasetDict
import evaluate

ModuleNotFoundError: No module named 'evaluate'